# SAM3 Stage-1: Finetune TinyViT via Encoder Distillation (Colab-ready)

Self-contained notebook — **no local clone required**. Runs on Google Colab with a T4/A100, or locally with CPU/CUDA/MPS.

Pipeline (same recipe as `efficientsam3/stage1/train_image_encoder_stage1.py`):

1. **Teacher** — full SAM3 vision backbone (~462 M params). Downloaded from `facebook/sam3` on the Hugging Face Hub.
2. **Student** — TinyViT-{5m, 11m, 21m} + small conv head projecting the last feature map to the teacher's `(C=1024, H=72, W=72)` grid.
3. **Loss** — masked pixel-wise MSE + cosine loss on valid (non-padded) pixels.
4. **Output** — a `.pth` checkpoint compatible with `stage1/convert_image_encoder_weights_stage1.py`.

**Before you start:**

- Request access to [facebook/sam3](https://huggingface.co/facebook/sam3) on Hugging Face (it's gated).
- Runtime → Change runtime type → **GPU (T4 is fine for a smoke test, A100 for full runs)**.

## 1. Install deps + clone efficientsam3

We install only what's needed for the image-encoder distillation path (skip MobileCLIP / mmengine / decord). On Colab this takes ~30 s.

In [ ]:
# Colab comes with torch/torchvision/numpy/Pillow preinstalled. Add the few extras we need.
%pip install -q timm==1.0.17 iopath ftfy==6.1.1 regex huggingface_hub tensorboard

# Clone efficientsam3 once (skip if already cloned).
import os, sys, subprocess, pathlib

REPO_URL = 'https://github.com/SimonZeng7108/efficientsam3'
REPO_DIR = pathlib.Path('/content/efficientsam3') if pathlib.Path('/content').exists() else pathlib.Path.cwd() / 'efficientsam3_colab'

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)

print('repo at', REPO_DIR)
assert (REPO_DIR / 'stage1').is_dir() and (REPO_DIR / 'sam3').is_dir()

## 2. Hugging Face login (required for the teacher)

`facebook/sam3` is a gated model. Paste a token with **read** access; generate one at https://huggingface.co/settings/tokens. Skip this cell if you'll supply a local checkpoint via `SAM3_CKPT`.

In [ ]:
try:
    from huggingface_hub import notebook_login
    notebook_login()
except Exception as e:
    print('notebook_login unavailable, fall back to: huggingface-cli login')
    print(e)

## 3. Paths + config

Single place to tweak backbone size, batch size, epochs. Defaults are chosen for a Colab T4 smoke run.

In [ ]:
STAGE1 = REPO_DIR / 'stage1'
SAM3_PKG = REPO_DIR / 'sam3'

for p in (str(STAGE1), str(SAM3_PKG)):
    if p not in sys.path:
        sys.path.insert(0, p)

# --- training knobs ---
BACKBONE       = 'tiny_vit_5m'     # tiny_vit_5m | tiny_vit_11m | tiny_vit_21m
IMG_SIZE       = 1008              # SAM3 teacher native input
EMBED_DIM      = 1024              # SAM3 trunk channels
EMBED_SIZE     = 72                # 72 = 1008 / 14

BATCH_SIZE     = 2                 # T4: 2; A100: 8-16
EPOCHS         = 2                 # 30-50 for a real distill
NUM_SAMPLES    = 32                # -1 = use all images found

BASE_LR        = 1e-3
WARMUP_EPOCHS  = 1
WEIGHT_DECAY   = 0.01
CLIP_GRAD      = 5.0
COSINE_WEIGHT  = 1.0

# --- data ---
# Default: use the ~400 bundled frames in efficientsam3/sam3/assets/videos/0001/ for a smoke run.
# Override IMG_DIR with your own folder of jpg/png (e.g. an SA-1B subset) for a real run.
IMG_DIR        = REPO_DIR / 'sam3' / 'assets' / 'videos' / '0001'

# --- teacher ---
SAM3_CKPT      = None              # None = auto-download from facebook/sam3; or pass a local path

# --- output ---
OUTPUT_DIR = pathlib.Path('/content/output') if pathlib.Path('/content').exists() else pathlib.Path.cwd() / 'output'
OUTPUT_DIR = OUTPUT_DIR / 'stage1' / f'es_{BACKBONE}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- device ---
import torch
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')
USE_AMP = DEVICE.type == 'cuda'

print(f'device={DEVICE}  amp={USE_AMP}')
print(f'backbone={BACKBONE}  img={IMG_SIZE}  embed=(C={EMBED_DIM}, H=W={EMBED_SIZE})')
print(f'images = {IMG_DIR}')
print(f'output = {OUTPUT_DIR}')

## 4. Student — TinyViT + projection head

Inlined here to keep imports small (the version in `stage1/model.py` also pulls in MobileCLIP + tokenizer). Semantics match `TinyViTAdapter` + `ImageStudentEncoder` verbatim.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

from sam3.backbones.tiny_vit import tiny_vit_5m_224, tiny_vit_11m_224, tiny_vit_21m_224

_BACKBONE_FNS = {
    'tiny_vit_5m':  tiny_vit_5m_224,
    'tiny_vit_11m': tiny_vit_11m_224,
    'tiny_vit_21m': tiny_vit_21m_224,
}

class TinyViTAdapter(nn.Module):
    """Strip TinyViT's classification head and reshape the last token sequence back to NCHW."""
    def __init__(self, model, img_size):
        super().__init__()
        self.model = model
        self.model.head = nn.Identity()
        self.final_hw = self._compute_resolution(img_size)
        self.out_channels = self.model.norm_head.normalized_shape[0]
        self.model.norm_head = nn.Identity()

    def _compute_resolution(self, img_size):
        H, W = self.model.patches_resolution
        for _ in range(self.model.num_layers - 1):
            H = (H - 1) // 2 + 1
            W = (W - 1) // 2 + 1
        return (H, W)

    def forward(self, x):
        x = self.model.patch_embed(x)
        for layer in self.model.layers:
            x = layer(x)
        B, N, C = x.shape
        H, W = self.final_hw
        return x.view(B, H, W, C).permute(0, 3, 1, 2).contiguous()


class ImageStudentEncoder(nn.Module):
    def __init__(self, backbone, in_channels, embed_dim, embed_size, img_size):
        super().__init__()
        self.backbone = backbone
        self.embed_size = embed_size
        self.img_size = img_size
        self.head = nn.Sequential(
            nn.Conv2d(in_channels, embed_dim, kernel_size=1, bias=False),
            nn.BatchNorm2d(embed_dim),
            nn.GELU(),
            nn.Conv2d(embed_dim, embed_dim, kernel_size=3, padding=1),
        )

    def forward(self, x):
        feats = self.backbone(x)
        feats = self.head(feats)
        if feats.shape[-1] != self.embed_size or feats.shape[-2] != self.embed_size:
            feats = F.interpolate(feats, size=(self.embed_size, self.embed_size), mode='bilinear', align_corners=False)
        return feats


def build_student(backbone_name, img_size, embed_dim, embed_size, pretrained_timm=False):
    backbone = _BACKBONE_FNS[backbone_name](pretrained=pretrained_timm, img_size=img_size)
    adapter  = TinyViTAdapter(backbone, img_size)
    return ImageStudentEncoder(adapter, adapter.out_channels, embed_dim, embed_size, img_size)

student = build_student(BACKBONE, IMG_SIZE, EMBED_DIM, EMBED_SIZE, pretrained_timm=False).to(DEVICE)
n_params = sum(p.numel() for p in student.parameters() if p.requires_grad)
print(f'student params: {n_params/1e6:.2f} M')

with torch.no_grad():
    y = student(torch.zeros(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE))
print('student output shape:', tuple(y.shape))
assert y.shape == (1, EMBED_DIM, EMBED_SIZE, EMBED_SIZE)

## 5. Teacher — SAM3 vision backbone (frozen)

Uses `build_sam3_image_model(..., enable_text_encoder=False)` which skips the 354 M text encoder. With `SAM3_CKPT=None` the checkpoint is pulled from `facebook/sam3` on first call.

In [ ]:
from sam3.model_builder import build_sam3_image_model

class SAM3ImageTeacher(nn.Module):
    """Wrap SAM3's ViT trunk so it returns a single (B, 1024, H, W) feature map."""
    def __init__(self, checkpoint_path=None, embed_size=72):
        super().__init__()
        self.embed_size = embed_size
        self.sam3 = build_sam3_image_model(
            checkpoint_path=checkpoint_path,
            load_from_HF=(checkpoint_path is None),
            eval_mode=True, device='cpu',
            enable_segmentation=True, enable_inst_interactivity=False, compile=False,
            enable_text_encoder=False,
        )
        for p in self.sam3.parameters():
            p.requires_grad = False
        self.sam3.eval()

    def forward(self, x):
        feats = self.sam3.backbone.vision_backbone.trunk(x)[-1]
        if feats.shape[-1] != self.embed_size or feats.shape[-2] != self.embed_size:
            feats = F.interpolate(feats, size=(self.embed_size, self.embed_size), mode='bilinear', align_corners=False)
        return feats

ckpt_arg = str(SAM3_CKPT) if SAM3_CKPT else None
print('building SAM3 teacher (this downloads ~3-5 GB on first run)...')
teacher = SAM3ImageTeacher(checkpoint_path=ckpt_arg, embed_size=EMBED_SIZE).to(DEVICE).eval()
print(f'teacher params: {sum(p.numel() for p in teacher.parameters())/1e6:.1f} M (frozen)')

with torch.no_grad():
    ty = teacher(torch.zeros(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE))
print('teacher output shape:', tuple(ty.shape))
assert ty.shape == (1, EMBED_DIM, EMBED_SIZE, EMBED_SIZE)

## 6. Dataset — any folder of images

`FlatImageDataset` loads images from a folder, applies `ResizeLongestSide(IMG_SIZE)` + SAM3's RGB normalisation + pads to `IMG_SIZE × IMG_SIZE`. It also returns the pre-pad shape, which we use to mask the loss on valid pixels only.

In [ ]:
import glob
import numpy as np
from PIL import Image
from torchvision.transforms.functional import pil_to_tensor

PIXEL_MEAN = torch.tensor([123.675, 116.28, 103.53]).view(-1, 1, 1)
PIXEL_STD  = torch.tensor([58.395, 57.12, 57.375]).view(-1, 1, 1)

def resize_longest(img, target):
    # img: (3, H, W) float
    h, w = img.shape[-2:]
    scale = target / max(h, w)
    nh, nw = int(round(h * scale)), int(round(w * scale))
    return F.interpolate(img[None], size=(nh, nw), mode='bilinear', align_corners=False, antialias=True).squeeze(0)

class FlatImageDataset(torch.utils.data.Dataset):
    def __init__(self, root, img_size=IMG_SIZE, num_samples=-1, exts=('.jpg', '.jpeg', '.png')):
        self.img_size = img_size
        paths = []
        for ext in exts:
            paths += glob.glob(f'{root}/**/*{ext}', recursive=True)
        paths.sort()
        if num_samples > 0:
            paths = paths[:num_samples]
        self.paths = paths
        if not self.paths:
            raise FileNotFoundError(f'no images under {root}')

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        img = pil_to_tensor(img).float()
        img = resize_longest(img, self.img_size)
        pre_pad = torch.tensor(img.shape, dtype=torch.long)   # (3, h, w)
        img = (img - PIXEL_MEAN) / PIXEL_STD
        h, w = img.shape[-2:]
        img = F.pad(img, (0, self.img_size - w, 0, self.img_size - h))
        return img, pre_pad

def collate(batch):
    imgs   = torch.stack([b[0] for b in batch], dim=0)
    sizes  = [b[1] for b in batch]
    return imgs, sizes

dataset = FlatImageDataset(str(IMG_DIR), img_size=IMG_SIZE, num_samples=NUM_SAMPLES)
loader = torch.utils.data.DataLoader(
    dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2 if DEVICE.type == 'cuda' else 0,
    pin_memory=(DEVICE.type == 'cuda'),
    drop_last=False, collate_fn=collate,
)
print(f'dataset size = {len(dataset)}   batches/epoch = {len(loader)}')

## 7. Losses

Masked MSE + cosine, copied from `stage1/train_image_encoder_stage1.py`. The valid mask is built from the pre-pad image shape so padding pixels are ignored.

In [ ]:
def build_valid_mask(pre_pad_shapes, target_shape, device, img_size=IMG_SIZE):
    B = len(pre_pad_shapes)
    valid = torch.zeros(B, 1, img_size, img_size, device=device)
    for i, s in enumerate(pre_pad_shapes):
        h, w = int(s[1]), int(s[2])
        valid[i, :, :h, :w] = 1
    valid = F.interpolate(valid, size=target_shape[-2:], mode='bilinear', align_corners=False)
    return (valid > 0.5).float()

def masked_mse(preds, teacher, mask):
    diff = (preds - teacher) * mask
    denom = mask.sum(dim=(1, 2, 3)).clamp(min=1.0)
    return (diff.square().sum(dim=(1, 2, 3)) / denom).mean()

def masked_cosine_loss(preds, teacher, mask):
    sim = F.cosine_similarity(preds, teacher, dim=1)
    loss = 1.0 - sim
    mask = mask.squeeze(1)
    loss = loss * mask
    denom = mask.sum(dim=(1, 2)).clamp(min=1.0)
    return (loss.sum(dim=(1, 2)) / denom).mean()

## 8. Optimizer + LR schedule

AdamW with linear warmup + cosine decay — same shape as `base_stage1.yaml`, stepped per iteration so short runs still see warmup.

In [ ]:
import math

optimizer = torch.optim.AdamW(
    student.parameters(), lr=BASE_LR, betas=(0.9, 0.999), eps=1e-8, weight_decay=WEIGHT_DECAY,
)

steps_per_epoch = max(1, len(loader))
total_steps   = EPOCHS * steps_per_epoch
warmup_steps  = WARMUP_EPOCHS * steps_per_epoch
MIN_LR        = BASE_LR * 1e-2
WARMUP_LR     = BASE_LR * 1e-3

def lr_at(step):
    if step < warmup_steps:
        return WARMUP_LR + (BASE_LR - WARMUP_LR) * (step / max(1, warmup_steps))
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return MIN_LR + 0.5 * (BASE_LR - MIN_LR) * (1 + math.cos(math.pi * progress))

scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
print(f'total_steps={total_steps}  warmup_steps={warmup_steps}')

## 9. Train

Online distillation: each step runs the teacher (no grad) + student + loss + backward. Fine for small datasets / a smoke run; for multi-epoch SA-1B training, pre-cache teacher embeddings via `stage1/scripts/save_image_embeddings.sh` and swap the inner loop to read from disk.

In [ ]:
import time

global_step = 0
history = []
start = time.time()

for epoch in range(EPOCHS):
    student.train()
    ep_loss, ep_n = 0.0, 0
    for i, (imgs, sizes) in enumerate(loader):
        imgs = imgs.to(DEVICE, non_blocking=True)

        lr = lr_at(global_step)
        for g in optimizer.param_groups:
            g['lr'] = lr

        with torch.no_grad():
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                t_feat = teacher(imgs)

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            s_feat = student(imgs)
            mask = build_valid_mask(sizes, s_feat.shape, s_feat.device)
            loss = masked_mse(s_feat, t_feat.to(s_feat.dtype), mask)
            if COSINE_WEIGHT > 0.0:
                loss = loss + COSINE_WEIGHT * masked_cosine_loss(s_feat, t_feat.to(s_feat.dtype), mask)

        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        if CLIP_GRAD is not None:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(student.parameters(), CLIP_GRAD)
        scaler.step(optimizer)
        scaler.update()

        lv = loss.detach().item()
        ep_loss += lv * imgs.size(0)
        ep_n    += imgs.size(0)
        history.append({'step': global_step, 'epoch': epoch, 'loss': lv, 'lr': lr})
        global_step += 1

        if i % 5 == 0:
            print(f'[ep {epoch:02d}  step {global_step:04d}/{total_steps}]  lr={lr:.2e}  loss={lv:.4f}')

    print(f'== epoch {epoch} mean loss = {ep_loss/max(1,ep_n):.4f} ==')

print(f'total time: {time.time()-start:.1f}s')

## 10. Save checkpoint

Format matches `stage1/convert_image_encoder_weights_stage1.py` (expects `ckpt['model']` = student `state_dict`). After the run, download the file from `/content/output/...` or mount Google Drive to persist it.

In [ ]:
ckpt_path = OUTPUT_DIR / f'ckpt_epoch_{EPOCHS-1}.pth'
torch.save({
    'model': student.state_dict(),
    'backbone': BACKBONE,
    'img_size': IMG_SIZE,
    'embed_dim': EMBED_DIM,
    'embed_size': EMBED_SIZE,
    'epoch': EPOCHS - 1,
}, ckpt_path)
print('saved', ckpt_path)
print('\nmerge into full SAM3:')
print(f'  python {STAGE1}/convert_image_encoder_weights_stage1.py \\\n'
      f'    --student-ckpt {ckpt_path} \\\n'
      f'    --sam3-ckpt <path/to/sam3.pt> \\\n'
      f'    --output {OUTPUT_DIR / f"efficient_sam3_{BACKBONE}.pt"}')

## 11. (Optional) Persist to Google Drive

On Colab, runtime storage vanishes when the session ends. Mount Drive and copy the checkpoint over.

In [ ]:
# Uncomment on Colab:
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# dest = pathlib.Path('/content/drive/MyDrive/sam3_stage1') / ckpt_path.name
# dest.parent.mkdir(parents=True, exist_ok=True)
# shutil.copy(ckpt_path, dest)
# print('copied to', dest)

## 12. Scaling up

- **Real distillation on SA-1B**: set `IMG_DIR` to a folder with tens of thousands of images (`data/sa-1b-1p.txt` gives download links for a 1 % subset, ~120 K images). Bump `EPOCHS` to 30–50 and `BATCH_SIZE` to whatever fits in VRAM.
- **Cache the teacher**: online distillation burns 462 M params of compute every step. For multi-epoch training, run the teacher once and write embeddings to disk — see `stage1/scripts/save_image_embeddings.sh` and `stage1/data/augmentation/dataset_wrapper.py`.
- **Pretrained init**: flip `pretrained_timm=True` in `build_student` to start from ImageNet-trained TinyViT weights.
- **Bigger students**: swap `BACKBONE` to `tiny_vit_11m` (~10 M) or `tiny_vit_21m` (~21 M).